In [ ]:
# Базовые библиотеки для воспроизводимости, анализа и удобного вывода результатов.
import os
import re
import sys
import random
import subprocess
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display, Markdown
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors

os.environ["TOKENIZERS_PARALLELISM"] = "false"


def safe_ensure_package(package_name: str, import_name: Optional[str] = None) -> bool:
    """Пытается импортировать пакет и при необходимости установить его через pip.
    Если установка не удалась, возвращает False, но не роняет ноутбук.
    """
    target = import_name or package_name
    try:
        __import__(target)
        return True
    except Exception:
        print(f"Пробуем установить пакет: {package_name}")
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package_name])
            __import__(target)
            return True
        except Exception as e:
            print(f"Не удалось подготовить пакет {package_name}: {e!r}")
            return False


FAISS_READY = safe_ensure_package("faiss-cpu", "faiss")

try:
    import faiss  # type: ignore
except Exception:
    faiss = None
    FAISS_READY = False


# sentence-transformers опционален: ноутбук умеет работать и без него.
SENTENCE_TRANSFORMERS_READY = safe_ensure_package("sentence-transformers", "sentence_transformers")

print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("FAISS доступен:", FAISS_READY)
print("sentence-transformers доступен:", SENTENCE_TRANSFORMERS_READY)

SyntaxError: invalid syntax (577607852.py, line 58)

In [2]:
# Фиксируем seed и определяем устройство.
RANDOM_STATE = 42
def set_seed(seed: int = RANDOM_STATE) -> None:
    random.seed(seed)
    np.random.seed(seed)


set_seed(RANDOM_STATE)

try:
    import torch
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
except Exception:
    DEVICE = "cpu"

print("Устройство для работы:", DEVICE)

Устройство для работы: cpu


#### Документы (База знаний)

In [3]:
# Небольшой корпус документов c FAQ 
# Данные были взяты с https://www.mirea.ru/education/voennyy-uchebnyy-tsentr/postupayushchim/voprosy-i-otvety/
documents: List[Dict[str,str]] =[
    {
        "doc_id": "doc_01",
        "title": "Когда студент поступает в ВУЦ?",
        "text": ("В ВУЦ РТУ МИРЭА поступают студенты 2-го курса очной формы обучения по основной образовательной программе. "
        "Обучение начинается с 3-го курса, при этом все мероприятия отбора проводятся в 4-м семестре (на 2-м курсе)."
        )
    },
        {
        "doc_id": "doc_02",
        "title": "Можно ли получить дополнительные баллы при поступлении в ВУЦ?",
        "text": ("Возможность получить дополнительные баллы при поступлении в ВУЦ не предусматривается. "
                "Баллы начисляются только за оценку текущей успеваемости и оценку уровня физической подготовки."
        )
    },
    {
        "doc_id": "doc_03",
        "title": "Какие и у кого есть преимущества при поступлении?",
        "text": ("Право допуска к обучению по программам подготовки запаса"
        " имеют граждане – дети военнослужащих, погибших или получивших увечье (ранение, травму, контузию)"
        " либо заболевание при исполнении обязанностей военной службы в ходе специальной военной операции, "
        "допущенные к конкурсному отбору и соответствующие требованиям по уровню физической подготовки."
        )
    },
    {
        "doc_id": "doc_04",
        "title": "Сколько длится обучение в Военном учебном центре РТУ МИРЭА?",
        "text": ("В зависимости от программ военной подготовки:"
        "Офицеры запаса — 3 года (6 семестров);"
        "Сержанты запаса — 2,5 года (5 семестров);"
        "Солдаты запаса — 2 года (4 семестров)"
        )
    },
    {
        "doc_id": "doc_05",
        "title": "Что делать, если по результатам медицинского освидетельствования установлена категория В, Г или 4-я группа профпригодности?",
        "text": ("Если по результатам медицинского обследования установлена категория В или Г, "
        "а также если у кандидата 4-я категория профессиональной пригодности,"
        " то проходить дальнейшие мероприятия не надо."
        )
    },
    {
        "doc_id": "doc_06",
        "title": "Нужно ли служить в ВС РФ после окончания ВУЦ?",
        "text": ("По окончании обучения в ВУЦ по программам подготовки солдат запаса,"
        " сержантов запаса или офицеров запаса необходимости в обязательном прохождении"
        " военной службы по призыву не будет. "
        "В этом случае студенту присваивается воинское звание, и он зачисляется в запас."
        )
    },
    {
        "doc_id": "doc_07",
        "title": "В случае отчисления из ВУЦ надо ли компенсировать затраты на обучение?",
        "text": ("При обучении по программам подготовки солдат запаса,"
        " сержантов запаса или офицеров запаса в случае отчисления из ВУЦ"
        " компенсировать затраты на обучение не нужно."
        " При этом студент продолжает обучение в университете."
        )
    },
    {
        "doc_id": "doc_08",
        "title": "Куда необходимо подать заявление для поступления?",
        "text": ("В личном кабинете студентов 2-го и 3-го курсов"
        " появится возможность заполнения электронной формы заявления об участии в конкурсном отборе"
        " для допуска к военной подготовке."
        " Заявление необходимо заполнить с 10 по 28 марта."
        )
    },
    {
        "doc_id": "doc_09",
        "title": "Что делать, если студент меняет паспорт в момент поступления?",
        "text": ("При замене паспорта на этапе подачи заявления допускается использование копии имеющегося паспорта. "
        "При подаче документов указывайте данные старого паспорта, "
        "а после, как получите новый, предоставьте в ВУЦ сведения о нём."
        )
    },
    {
        "doc_id": "doc_10",
        "title": "Как сдать нормативы по физической подготовке?",
        "text": ("Со студентами, успешно прошедшими предварительный отбор,"
        " с 15 мая по 14 июня 2026 года проводится оценка физической подготовки."
        " Даты и место проведения оценки физической подготовки,"
        " а также порядок записи будут сообщены после 10 мая 2026 года."
        " Студент прибывает на оценку физической подготовки в спортивной форме одежды"
        " с документом, удостоверяющим личность, и паспортом физической подготовки."
        )
    },
    {
        "doc_id": "doc_11",
        "title": "Кто имеет право сдавать нормативы по физической подготовке?",
        "text": ("До сдачи нормативов по физической подготовке допускаются граждане,"
        " прошедшие медицинское освидетельствование, "
        "профессиональный психологический отбор и допущенные по состоянию здоровья."
        )
    },
    {
        "doc_id": "doc_12",
        "title": "Принцип формирования протоколов для допуска к военной подготовке?",
        "text": ("Протокол конкурсного отбора граждан для допуска к военной подготовке в ВУЦ "
        "формируется по количеству наибольших баллов. "
        "У кого баллов больше, тот находится вверху списка, и далее по убывающей."
        )
    },
    {
        "doc_id": "doc_13",
        "title": "Что делать, если заболел на момент сдачи нормативов по физической подготовке?",
        "text": ("Вы можете сдать нормативы по физической подготовке в другой день, "
        "определённый кафедрой физического воспитания."
        )
    },
    {
        "doc_id": "doc_14",
        "title": "Как перезаписаться на сдачу нормативов по физической подготовке?",
        "text": ("Чтобы заново записаться на сдачу нормативов, "
        "направьте письмо на тот же адрес электронной почты, что использовался при записи."
        " Перезапись возможна, только если кандидат по каким-либо причинам "
        "не смог прибыть на место сдачи нормативов и не сдавал их. "
        "Пересдача нормативов с целью повышения оценки не допускается."
        )
    },
    {
        "doc_id": "doc_15",
        "title": "Категории годности, с которыми граждане допускаются для обучения в ВУЦ?",
        "text": ("Для допуска к военной подготовке в ВУЦ кандидат должен "
                 "иметь категории годности к военной службе: А (А1, А2…) или Б (Б1, Б2, Б3…)."
                )
    },
    {
        "doc_id": "doc_16",
        "title": "Где гражданину получить справку по форме № 2?",
        "text": ("Справка по форме № 2 выдается отделом воинского учёта обучающихся и бронирования в кабинете А-312 по адресу:"
                " проспект Вернадского, 78 или Стромынка, 20, кабинет А-302. "
                "ВУЦ воинским учётом не занимается и подобных справок не выдаёт."
                 )
    },
    {
        "doc_id": "doc_17",
        "title": "Принцип формирования списков поступающих в ВУЦ (рейтинга)?",
        "text": ("Решение о допуске к военной подготовке в Военном учебном центре принимается"
        " в зависимости от итоговой суммы баллов. "
        "Баллы за сдачу физической подготовки и текущей успеваемости учитываются одинаково."
        )
    },
    {
        "doc_id": "doc_18",
        "title": "Обязательно ли поступление в магистратуру при обучении в ВУЦ?",
        "text": ("При обучении по программам подготовки офицеров запаса и освоения направлений бакалавриата"
        " для окончания полного курса военной подготовки необходимо "
        "поступление в магистратуру РТУ МИРЭА на очную форму обучения."
        )
    },
    {
        "doc_id": "doc_19",
        "title": "Допускается ли приём граждан в ВУЦ из другого вуза, где нет военной подготовки?",
        "text": ("Приём граждан в ВУЦ из другого вуза, где нет военной подготовки, в РТУ МИРЭА не предусмотрен."
        )
    },
    {
        "doc_id": "doc_20",
        "title": "Куда обращаться при возникновении проблем?",
        "text": ("ВУЦ осуществляет только обработку документов. "
        "Решать проблему необходимо там, где она возникла. "
        "Например, по вопросам, связанным с состоянием здоровья, "
        "следует обращаться в военкомат по месту прохождения ВВК, "
        "а по поводу сдачи нормативов физической подготовки — на кафедру физического воспитания; "
        "по поводу оформления допуска к сведениям, "
        "составляющим государственную тайну — в первый отдел и т.д."
        )
    },
]

docs_df = pd.DataFrame(documents)
display(docs_df[["doc_id", "title"]])

,doc_id,title
0,doc_01,Когда студент поступает в ВУЦ?
1,doc_02,Можно ли получить дополнительные баллы при пос...
2,doc_03,Какие и у кого есть преимущества при поступлении?
3,doc_04,Сколько длится обучение в Военном учебном цент...
4,doc_05,"Что делать, если по результатам медицинского о..."
5,doc_06,Нужно ли служить в ВС РФ после окончания ВУЦ?
6,doc_07,В случае отчисления из ВУЦ надо ли компенсиров...
7,doc_08,Куда необходимо подать заявление для поступления?
8,doc_09,"Что делать, если студент меняет паспорт в моме..."
9,doc_10,Как сдать нормативы по физической подготовке?


#### Чанкинг

In [4]:
# Простая функция чанкинга по словам.
def chunk_text(text: str, chunk_size: int = 22, overlap: int = 5) -> List[str]:
    words = text.replace("\n", " ").split()

    if chunk_size <= 0:
        raise ValueError("chunk_size должен быть положительным.")
    if overlap >= chunk_size:
        raise ValueError("overlap должен быть меньше chunk_size.")

    chunks = []
    step = chunk_size - overlap

    for start in range(0, len(words), step):
        chunk_words = words[start : start + chunk_size]
        if not chunk_words:
            continue

        chunks.append(" ".join(chunk_words))

        if start + chunk_size >= len(words):
            break

    return chunks


def build_chunks_dataframe(
    docs: List[Dict[str, str]],
    chunk_size: int = 22,
    overlap: int = 5,
) -> pd.DataFrame:
    rows = []

    for doc in docs:
        chunks = chunk_text(doc["text"], chunk_size=chunk_size, overlap=overlap)
        for chunk_id, chunk in enumerate(chunks):
            rows.append(
                {
                    "doc_id": doc["doc_id"],
                    "title": doc["title"],
                    "chunk_id": chunk_id,
                    "chunk_text": chunk,
                    "n_words": len(chunk.split()),
                }
            )

    return pd.DataFrame(rows)


chunks_df = build_chunks_dataframe(documents, chunk_size=22, overlap=5)

print("Количество чанков:", len(chunks_df))
display(chunks_df.head(10))
chunks_df.to_csv("artifacts/chunk_examples.csv")

Количество чанков: 40


,doc_id,title,chunk_id,chunk_text,n_words
0,doc_01,Когда студент поступает в ВУЦ?,0,В ВУЦ РТУ МИРЭА поступают студенты 2-го курса ...,22
1,doc_01,Когда студент поступает в ВУЦ?,1,"с 3-го курса, при этом все мероприятия отбора ...",15
2,doc_02,Можно ли получить дополнительные баллы при пос...,0,Возможность получить дополнительные баллы при ...,22
3,doc_03,Какие и у кого есть преимущества при поступлении?,0,Право допуска к обучению по программам подгото...,22
4,doc_03,Какие и у кого есть преимущества при поступлении?,1,"(ранение, травму, контузию) либо заболевание п...",22
5,doc_03,Какие и у кого есть преимущества при поступлении?,2,конкурсному отбору и соответствующие требовани...,9
6,doc_04,Сколько длится обучение в Военном учебном цент...,0,В зависимости от программ военной подготовки:О...,22
7,doc_04,Сколько длится обучение в Военном учебном цент...,1,семестров);Солдаты запаса — 2 года (4 семестров),7
8,doc_05,"Что делать, если по результатам медицинского о...",0,Если по результатам медицинского обследования ...,22
9,doc_05,"Что делать, если по результатам медицинского о...",1,"профессиональной пригодности, то проходить дал...",8


Данные chunk_size и overlap были выбраны из расчета, что наша база знаний состоит из маленьких текстов.

#### Embeding

In [5]:
# Единый интерфейс для двух вариантов векторизации: dense embeddings и fallback.
class EmbeddingBackend:
    def fit_documents(self, texts: List[str]) -> np.ndarray:
        raise NotImplementedError

    def encode_queries(self, texts: List[str]) -> np.ndarray:
        raise NotImplementedError


class SentenceTransformersBackend(EmbeddingBackend):
    def __init__(self, model_name: str, device: str = "cpu") -> None:
        from sentence_transformers import SentenceTransformer  # type: ignore

        self.model_name = model_name
        self.model = SentenceTransformer(model_name, device=device)
        self.backend_name = f"SentenceTransformer: {model_name}"

    def fit_documents(self, texts: List[str]) -> np.ndarray:
        vectors = self.model.encode(
            texts,
            batch_size=16,
            show_progress_bar=False,
            convert_to_numpy=True,
            normalize_embeddings=True,
        )
        return vectors.astype("float32")

    def encode_queries(self, texts: List[str]) -> np.ndarray:
        vectors = self.model.encode(
            texts,
            batch_size=16,
            show_progress_bar=False,
            convert_to_numpy=True,
            normalize_embeddings=True,
        )
        return vectors.astype("float32")


class TfidfFallbackBackend(EmbeddingBackend):
    def __init__(self) -> None:
        self.vectorizer = TfidfVectorizer(ngram_range=(1, 2), lowercase=True)
        self.backend_name = "TF-IDF fallback"

    def fit_documents(self, texts: List[str]) -> np.ndarray:
        vectors = self.vectorizer.fit_transform(texts).toarray()
        return vectors.astype("float32")

    def encode_queries(self, texts: List[str]) -> np.ndarray:
        vectors = self.vectorizer.transform(texts).toarray()
        return vectors.astype("float32")


def build_embedding_backend(
    model_name: str = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    device: str = "cpu",
) -> EmbeddingBackend:
    try:
        backend = SentenceTransformersBackend(model_name=model_name, device=device)
        print("Используем полноценные dense embeddings.")
        print("Бэкэнд:", backend.backend_name)
        return backend
    except Exception as e:
        print("Не удалось загрузить sentence-transformers encoder.")
        print("Причина:", repr(e))
        print("Переключаемся на TF-IDF fallback. Ноутбук останется рабочим,")
        print("но это уже не полноценные dense embeddings.")
        return TfidfFallbackBackend()


embedder = build_embedding_backend(device=DEVICE)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Используем полноценные dense embeddings.
Бэкэнд: SentenceTransformer: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2


#### Векторное представление чанков

In [6]:
# Строим векторные представления для всех чанков.
chunk_texts = chunks_df["chunk_text"].tolist()
chunk_embeddings = embedder.fit_documents(chunk_texts)

print("Форма матрицы эмбеддингов:", chunk_embeddings.shape)

# Проверяем длины векторов.
# Если normalize_embeddings=True сработал корректно, все нормы должны быть ≈ 1.0.
# Это означает, что косинусное сходство далее можно считать через скалярное произведение.
vector_norms = np.linalg.norm(chunk_embeddings, axis=1)
print("Минимальная норма:", round(float(vector_norms.min()), 4))
print("Максимальная норма:", round(float(vector_norms.max()), 4))
print("Средняя норма:", round(float(vector_norms.mean()), 4))
print("→ Нормы ≈ 1.0: нормировка подтверждена, dot product = cosine similarity.")

Форма матрицы эмбеддингов: (40, 384)
Минимальная норма: 1.0
Максимальная норма: 1.0
Средняя норма: 1.0
→ Нормы ≈ 1.0: нормировка подтверждена, dot product = cosine similarity.


#### Индекс `FAISS` для быстрого поиска

In [7]:
# Единая обёртка над FAISS и fallback-поиском.
class VectorSearchIndex:
    def __init__(self, dim: int) -> None:
        self.dim = dim
        self.backend_name = None
        self._faiss_index = None
        self._nn_index = None

        if FAISS_READY:
            self._faiss_index = faiss.IndexFlatIP(dim)  # type: ignore[name-defined]
            self.backend_name = "FAISS IndexFlatIP"
        else:
            self._nn_index = NearestNeighbors(metric="cosine")
            self.backend_name = "sklearn NearestNeighbors fallback"

    def add(self, vectors: np.ndarray) -> None:
        vectors = vectors.astype("float32")

        if self._faiss_index is not None:
            self._faiss_index.add(vectors)
        else:
            self._nn_index.fit(vectors)

    def search(self, query_vectors: np.ndarray, top_k: int = 5) -> Tuple[np.ndarray, np.ndarray]:
        query_vectors = query_vectors.astype("float32")

        if self._faiss_index is not None:
            scores, indices = self._faiss_index.search(query_vectors, top_k)
            return scores, indices

        distances, indices = self._nn_index.kneighbors(query_vectors, n_neighbors=top_k)
        scores = 1.0 - distances
        return scores, indices


search_index = VectorSearchIndex(dim=chunk_embeddings.shape[1])
search_index.add(chunk_embeddings)

print("Индекс построен.")
print("Бэкэнд индекса:", search_index.backend_name)

Индекс построен.
Бэкэнд индекса: FAISS IndexFlatIP


In [ ]:
# Удобная функция для поиска похожих чанков.
def search_similar_chunks(query: str, top_k: int = 5) -> pd.DataFrame:
    query_vectors = embedder.encode_queries([query])
    scores, indices = search_index.search(query_vectors, top_k=top_k)

    rows = []
    for rank, (score, idx) in enumerate(zip(scores[0], indices[0]), start=1):
        chunk_row = chunks_df.iloc[int(idx)]
        rows.append(
            {
                "rank": rank,
                "doc_id": chunk_row["doc_id"],
                "title": chunk_row["title"],
                "chunk_id": int(chunk_row["chunk_id"]),
                "score": round(float(score), 4),
                "chunk_text": chunk_row["chunk_text"],
            }
        )

    return pd.DataFrame(rows)


# Проверяем FAISS-поиск на первых запросах.
example_queries = ["Как получить дополнительные баллы?", "Что делать, если у меня категория годности В?", "Какие нормативы надо сдать?"]

for current_query in example_queries:
    display(Markdown(f"### Запрос: `{current_query}`"))
    display(search_similar_chunks(current_query, top_k=3))

### Запрос: `Как получить дополнительные баллы?`

,rank,doc_id,title,chunk_id,score,chunk_text
0,1,doc_12,Принцип формирования протоколов для допуска к ...,1,0.5768,"кого баллов больше, тот находится вверху списк..."
1,2,doc_02,Можно ли получить дополнительные баллы при пос...,0,0.5038,Возможность получить дополнительные баллы при ...
2,3,doc_17,Принцип формирования списков поступающих в ВУЦ...,0,0.3402,Решение о допуске к военной подготовке в Военн...


### Запрос: `Что делать, если у меня категория годности В?`

,rank,doc_id,title,chunk_id,score,chunk_text
0,1,doc_05,"Что делать, если по результатам медицинского о...",0,0.4726,Если по результатам медицинского обследования ...
1,2,doc_15,"Категории годности, с которыми граждане допуск...",0,0.3595,Для допуска к военной подготовке в ВУЦ кандида...
2,3,doc_03,Какие и у кого есть преимущества при поступлении?,1,0.2664,"(ранение, травму, контузию) либо заболевание п..."


### Запрос: `Какие нормативы надо сдать?`

,rank,doc_id,title,chunk_id,score,chunk_text
0,1,doc_11,Кто имеет право сдавать нормативы по физическо...,0,0.5616,До сдачи нормативов по физической подготовке д...
1,2,doc_20,Куда обращаться при возникновении проблем?,1,0.5116,"состоянием здоровья, следует обращаться в воен..."
2,3,doc_14,Как перезаписаться на сдачу нормативов по физи...,2,0.4790,не сдавал их. Пересдача нормативов с целью пов...


#### Контрольные запросы

In [9]:
# Набор контрольных запросов и ожидаемых релевантных документов.
benchmark_queries: List[Dict[str, object]] = [
    {   "query_id": "q01",
        "query": "На каком курсе можно поступить в ВУЦ?",
        "relevant_doc_ids": ["doc_01"],
    },
    {   "query_id": "q02",
        "query": "Как получить дополнительные баллы при поступлении?",
        "relevant_doc_ids": ["doc_02"],
    },
    {   "query_id": "q03",
        "query": "Есть ли льготы для детей военнослужащих при поступлении?",
        "relevant_doc_ids": ["doc_03"],
    },
    {   "query_id": "q04",
        "query": "Сколько лет учатся в ВУЦ?",
        "relevant_doc_ids": ["doc_04"],
    },
    {   "query_id": "q05",
        "query": "Что будет, если категория годности не подходит (например В)?",
        "relevant_doc_ids": ["doc_05"],
    },
    {   "query_id": "q06",
        "query": "Нужно ли потом идти служить после окончания ВУЦ?",
        "relevant_doc_ids": ["doc_06"],
    },
    {   "query_id": "q07",
        "query": "Придется ли платить, если меня отчислят из ВУЦ?",
        "relevant_doc_ids": ["doc_07"],
    },
    {   "query_id": "q08",
        "query": "Куда подавать заявление на поступление в ВУЦ?",
        "relevant_doc_ids": ["doc_08"],
    },
    {   "query_id": "q09",
        "query": "Что делать если я меняю паспорт во время подачи документов?",
        "relevant_doc_ids": ["doc_09"],
    },
    {   "query_id": "q10",
        "query": "Как проходит сдача нормативов по физической подготовке?",
        "relevant_doc_ids": ["doc_10"],
    },
    {   "query_id": "q11",
        "query": "Кто допускается до сдачи физподготовки?",
        "relevant_doc_ids": ["doc_11"],
    },
    {   "query_id": "q12",
        "query": "Как формируется рейтинг поступающих в ВУЦ?",
        "relevant_doc_ids": ["doc_17"],
    },
    {   "query_id": "q13",
        "query": "Если я заболел и не пришел на нормативы, можно ли пересдать?",
        "relevant_doc_ids": ["doc_13"],
    },
    {   "query_id": "q14",
        "query": "Можно ли поступить в ВУЦ из другого университета?",
        "relevant_doc_ids": ["doc_19"],
    }
]

benchmark_df = pd.DataFrame(benchmark_queries)
display(benchmark_df)

,query_id,query,relevant_doc_ids
0,q01,На каком курсе можно поступить в ВУЦ?,[doc_01]
1,q02,Как получить дополнительные баллы при поступле...,[doc_02]
2,q03,Есть ли льготы для детей военнослужащих при по...,[doc_03]
3,q04,Сколько лет учатся в ВУЦ?,[doc_04]
4,q05,"Что будет, если категория годности не подходит...",[doc_05]
5,q06,Нужно ли потом идти служить после окончания ВУЦ?,[doc_06]
6,q07,"Придется ли платить, если меня отчислят из ВУЦ?",[doc_07]
7,q08,Куда подавать заявление на поступление в ВУЦ?,[doc_08]
8,q09,Что делать если я меняю паспорт во время подач...,[doc_09]
9,q10,Как проходит сдача нормативов по физической по...,[doc_10]


#### Сборка retrieval-конвейер

In [10]:
@dataclass
class RetrievalArtifacts:
    backend_name: str
    backend: EmbeddingBackend
    chunks_df: pd.DataFrame
    chunk_vectors: np.ndarray
    index: object


def build_retriever(
    docs: List[Dict[str, str]],
    chunk_size: int = 22,
    overlap: int = 5,
    device: str = "cpu",
) -> RetrievalArtifacts:
    chunks = build_chunks_dataframe(docs, chunk_size=chunk_size, overlap=overlap)
    chunks_df = pd.DataFrame(chunks)

    backend = build_embedding_backend(device=device)
    chunk_vectors = backend.fit_documents(chunks_df["chunk_text"].tolist())

    if not FAISS_READY:
        raise RuntimeError("FAISS недоступен. Для этого ноутбука ожидается установленный faiss-cpu.")

    index = faiss.IndexFlatIP(chunk_vectors.shape[1])
    index.add(chunk_vectors)

    return RetrievalArtifacts(
        backend_name=backend.backend_name,
        backend=backend,
        chunks_df=chunks_df,
        chunk_vectors=chunk_vectors,
        index=index,
    )


def search_chunks(
    query: str,
    artifacts: RetrievalArtifacts,
    top_k: int = 3,
) -> pd.DataFrame:
    query_vector = artifacts.backend.encode_queries([query]).astype("float32")
    scores, indices = artifacts.index.search(query_vector, top_k)

    rows = []
    for rank, (score, idx) in enumerate(zip(scores[0], indices[0]), start=1):
        chunk_row = artifacts.chunks_df.iloc[int(idx)]
        rows.append(
            {
                "rank": rank,
                "score": float(score),
                "doc_id": chunk_row["doc_id"],
                "title": chunk_row["title"],
                "chunk_id": chunk_row["chunk_id"],
                "chunk_text": chunk_row["chunk_text"],
            }
        )
    return pd.DataFrame(rows)


def unique_doc_order(result_df: pd.DataFrame) -> List[str]:
    seen = set()
    ordered = []
    for doc_id in result_df["doc_id"].tolist():
        if doc_id not in seen:
            seen.add(doc_id)
            ordered.append(doc_id)
    return ordered


def evaluate_query(
    query: str,
    relevant_doc_ids: List[str],
    artifacts: RetrievalArtifacts,
    top_k: int = 3,
) -> Dict[str, object]:
    result_df = search_chunks(query, artifacts=artifacts, top_k=top_k)
    predicted_doc_ids = unique_doc_order(result_df)

    hit = int(any(doc_id in predicted_doc_ids for doc_id in relevant_doc_ids))
    recall = sum(doc_id in predicted_doc_ids for doc_id in relevant_doc_ids) / len(relevant_doc_ids)

    first_relevant_rank = None
    for idx, doc_id in enumerate(predicted_doc_ids, start=1):
        if doc_id in relevant_doc_ids:
            first_relevant_rank = idx
            break

    mrr = 0.0 if first_relevant_rank is None else 1.0 / first_relevant_rank

    return {
        "predicted_doc_ids": predicted_doc_ids,
        "hit": hit,
        "recall": recall,
        "first_relevant_rank": first_relevant_rank,
        "mrr": mrr,
        "result_df": result_df,
    }


def evaluate_benchmark(
    benchmark_rows: List[Dict[str, object]],
    artifacts: RetrievalArtifacts,
    top_k: int = 3,
) -> pd.DataFrame:
    rows = []
    for row in benchmark_rows:
        metrics = evaluate_query(
            query=row["query"],
            relevant_doc_ids=row["relevant_doc_ids"],
            artifacts=artifacts,
            top_k=top_k,
        )
        rows.append(
            {
                "query_id": row["query_id"],
                "query": row["query"],
                "relevant_doc_ids": ", ".join(row["relevant_doc_ids"]),
                "predicted_doc_ids": ", ".join(metrics["predicted_doc_ids"]),
                f"hit@{top_k}": metrics["hit"],
                f"recall@{top_k}": metrics["recall"],
                f"MRR@{top_k}": metrics["mrr"],
                "first_relevant_rank": metrics["first_relevant_rank"],
            }
        )
    return pd.DataFrame(rows)

#### Оценка retrieval

In [11]:
# Собираем baseline-конфигурацию retriever.
baseline_chunk_size = 22
baseline_overlap = 5

artifacts = build_retriever(
    documents,
    chunk_size=baseline_chunk_size,
    overlap=baseline_overlap,
    device=DEVICE,
)

print("Используемый backend:", artifacts.backend_name)
print("Количество чанков:", len(artifacts.chunks_df))
display(artifacts.chunks_df.head())

# Прогоняем весь benchmark и считаем метрики для baseline-конфигурации.
baseline_eval_k3 = evaluate_benchmark(benchmark_queries, artifacts=artifacts, top_k=3)
display(baseline_eval_k3)

summary_k3 = pd.DataFrame(
    {
        "metric": ["mean_hit@3", "mean_recall@3", "mean_MRR@3"],
        "value": [
            baseline_eval_k3["hit@3"].mean(),
            baseline_eval_k3["recall@3"].mean(),
            baseline_eval_k3["MRR@3"].mean(),
        ],
    }
)
display(summary_k3)

retrieval_eval = baseline_eval_k3.copy()
retrieval_eval = retrieval_eval.rename(columns={
    "relevant_doc_ids":"expected_source",
    "predicted_doc_ids":"retrieved_sources",
    "hit@3":"hit_at_k",
    "first_relevant_rank":"rank_of_first_relevant"})
retrieval_eval.to_csv("artifacts/retrieval_eval.csv")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Используем полноценные dense embeddings.
Бэкэнд: SentenceTransformer: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Используемый backend: SentenceTransformer: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Количество чанков: 40


,doc_id,title,chunk_id,chunk_text,n_words
0,doc_01,Когда студент поступает в ВУЦ?,0,В ВУЦ РТУ МИРЭА поступают студенты 2-го курса ...,22
1,doc_01,Когда студент поступает в ВУЦ?,1,"с 3-го курса, при этом все мероприятия отбора ...",15
2,doc_02,Можно ли получить дополнительные баллы при пос...,0,Возможность получить дополнительные баллы при ...,22
3,doc_03,Какие и у кого есть преимущества при поступлении?,0,Право допуска к обучению по программам подгото...,22
4,doc_03,Какие и у кого есть преимущества при поступлении?,1,"(ранение, травму, контузию) либо заболевание п...",22


,query_id,query,relevant_doc_ids,predicted_doc_ids,hit@3,recall@3,MRR@3,first_relevant_rank
0,q01,На каком курсе можно поступить в ВУЦ?,doc_01,"doc_18, doc_01, doc_08",1,1.0,0.5,2.0
1,q02,Как получить дополнительные баллы при поступле...,doc_02,"doc_02, doc_12, doc_17",1,1.0,1.0,1.0
2,q03,Есть ли льготы для детей военнослужащих при по...,doc_03,"doc_03, doc_06",1,1.0,1.0,1.0
3,q04,Сколько лет учатся в ВУЦ?,doc_04,"doc_04, doc_01, doc_18",1,1.0,1.0,1.0
4,q05,"Что будет, если категория годности не подходит...",doc_05,"doc_05, doc_14, doc_15",1,1.0,1.0,1.0
5,q06,Нужно ли потом идти служить после окончания ВУЦ?,doc_06,"doc_06, doc_04, doc_18",1,1.0,1.0,1.0
6,q07,"Придется ли платить, если меня отчислят из ВУЦ?",doc_07,"doc_07, doc_16, doc_06",1,1.0,1.0,1.0
7,q08,Куда подавать заявление на поступление в ВУЦ?,doc_08,"doc_16, doc_08, doc_06",1,1.0,0.5,2.0
8,q09,Что делать если я меняю паспорт во время подач...,doc_09,"doc_09, doc_14",1,1.0,1.0,1.0
9,q10,Как проходит сдача нормативов по физической по...,doc_10,"doc_20, doc_11, doc_03",0,0.0,0.0,NaN


,metric,value
0,mean_hit@3,0.785714
1,mean_recall@3,0.785714
2,mean_MRR@3,0.642857


#### Небольшой эксперимент с параметрами retrieval

In [12]:
# Проведем эксперимент с другими значениями chunk_size и overlap

artifacts_compare = build_retriever(
    documents,
    chunk_size=28,
    overlap=8,
    device=DEVICE,
)
print("Используемый backend:", artifacts_compare.backend_name)
print("Количество чанков:", len(artifacts_compare.chunks_df))
display(artifacts_compare.chunks_df.head())

# Прогоняем весь benchmark и считаем метрики для baseline-конфигурации.
baseline_eval_k3 = evaluate_benchmark(benchmark_queries, artifacts=artifacts_compare, top_k=3)
display(baseline_eval_k3)

summary_k3 = pd.DataFrame(
    {
        "metric": ["mean_hit@3", "mean_recall@3", "mean_MRR@3"],
        "value": [
            baseline_eval_k3["hit@3"].mean(),
            baseline_eval_k3["recall@3"].mean(),
            baseline_eval_k3["MRR@3"].mean(),
        ],
    }
)
display(summary_k3)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Используем полноценные dense embeddings.
Бэкэнд: SentenceTransformer: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Используемый backend: SentenceTransformer: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Количество чанков: 32


,doc_id,title,chunk_id,chunk_text,n_words
0,doc_01,Когда студент поступает в ВУЦ?,0,В ВУЦ РТУ МИРЭА поступают студенты 2-го курса ...,28
1,doc_01,Когда студент поступает в ВУЦ?,1,при этом все мероприятия отбора проводятся в 4...,12
2,doc_02,Можно ли получить дополнительные баллы при пос...,0,Возможность получить дополнительные баллы при ...,22
3,doc_03,Какие и у кого есть преимущества при поступлении?,0,Право допуска к обучению по программам подгото...,28
4,doc_03,Какие и у кого есть преимущества при поступлении?,1,либо заболевание при исполнении обязанностей в...,23


,query_id,query,relevant_doc_ids,predicted_doc_ids,hit@3,recall@3,MRR@3,first_relevant_rank
0,q01,На каком курсе можно поступить в ВУЦ?,doc_01,"doc_01, doc_07, doc_08",1,1.0,1.0,1.0
1,q02,Как получить дополнительные баллы при поступле...,doc_02,"doc_02, doc_12, doc_17",1,1.0,1.0,1.0
2,q03,Есть ли льготы для детей военнослужащих при по...,doc_03,"doc_03, doc_07, doc_06",1,1.0,1.0,1.0
3,q04,Сколько лет учатся в ВУЦ?,doc_04,"doc_01, doc_04, doc_07",1,1.0,0.5,2.0
4,q05,"Что будет, если категория годности не подходит...",doc_05,"doc_05, doc_14, doc_15",1,1.0,1.0,1.0
5,q06,Нужно ли потом идти служить после окончания ВУЦ?,doc_06,"doc_06, doc_04",1,1.0,1.0,1.0
6,q07,"Придется ли платить, если меня отчислят из ВУЦ?",doc_07,"doc_06, doc_16",0,0.0,0.0,NaN
7,q08,Куда подавать заявление на поступление в ВУЦ?,doc_08,"doc_20, doc_16",0,0.0,0.0,NaN
8,q09,Что делать если я меняю паспорт во время подач...,doc_09,"doc_09, doc_14, doc_16",1,1.0,1.0,1.0
9,q10,Как проходит сдача нормативов по физической по...,doc_10,"doc_11, doc_20, doc_13",0,0.0,0.0,NaN


,metric,value
0,mean_hit@3,0.642857
1,mean_recall@3,0.642857
2,mean_MRR@3,0.571429


#### Обновление базы знаний и переиндексация

In [13]:
new_documents: List[Dict[str, str]] = [
    {
        "doc_id": "doc_21",
        "title": "Что такое «Курс молодого бойца» и как он проходит?",
        "text": (
            "В соответствии с приказом РТУ МИРЭА прохождение начальной "
            "военной подготовки по программе «Курс молодого бойца» является обязательным."
            " Студенты, не прошедшие курс, к конкурсному отбору для обучения в ВУЦ РТУ МИРЭА не допускаются."
            "Курс будет проводиться на базе войсковой части Минобороны России. "
            "Место проведения курса будет сообщено позже."
        ),
    },
    {
        "doc_id": "doc_22",
        "title": "Как учитывается успеваемость и академическая задолженность?",
        "text": (
            "В целях оценки текущей успеваемости соответствующими учебными отделами институтов"
            " проводится расчёт среднего балла успеваемости (для студентов 2-го курса — за 2-й и 3-й семестры обучения,"
            " для студентов 3-го курса — за 4-й и 5-й семестры обучения) и занесение результатов в паспорт успеваемости."
        ),
    },
]

updated_documents = documents + new_documents


display(pd.DataFrame(new_documents)[["doc_id", "title"]])

new_queries = [
    "Что такое «Курс молодого бойца» и как он проходит?",
    "Как учитывается успеваемость и академическая задолженность?",
]

display(Markdown("### Как baseline-база отвечает на новые запросы"))
for query in new_queries:
    display(Markdown(f"**Запрос:** {query}"))
    display(search_chunks(query, artifacts=artifacts, top_k=3)[["rank", "score", "doc_id", "title", "chunk_text"]])

,doc_id,title
0,doc_21,Что такое «Курс молодого бойца» и как он прохо...
1,doc_22,Как учитывается успеваемость и академическая з...


### Как baseline-база отвечает на новые запросы

**Запрос:** Что такое «Курс молодого бойца» и как он проходит?

,rank,score,doc_id,title,chunk_text
0,1,0.419084,doc_17,Принцип формирования списков поступающих в ВУЦ...,Решение о допуске к военной подготовке в Военн...
1,2,0.399403,doc_04,Сколько длится обучение в Военном учебном цент...,семестров);Солдаты запаса — 2 года (4 семестров)
2,3,0.374579,doc_06,Нужно ли служить в ВС РФ после окончания ВУЦ?,обязательном прохождении военной службы по при...


**Запрос:** Как учитывается успеваемость и академическая задолженность?

,rank,score,doc_id,title,chunk_text
0,1,0.495745,doc_07,В случае отчисления из ВУЦ надо ли компенсиров...,компенсировать затраты на обучение не нужно. П...
1,2,0.462757,doc_18,Обязательно ли поступление в магистратуру при ...,необходимо поступление в магистратуру РТУ МИРЭ...
2,3,0.446645,doc_17,Принцип формирования списков поступающих в ВУЦ...,Решение о допуске к военной подготовке в Военн...


In [14]:
# Переиндексируем корпус уже с новыми документами.
updated_artifacts = build_retriever(
    updated_documents,
    chunk_size=baseline_chunk_size,
    overlap=baseline_overlap,
    device=DEVICE,
)

# Расширяем benchmark новыми запросами.
extended_benchmark_queries = benchmark_queries + [
    {
        "query_id": "q15",
        "query": "Что такое Курс молодого бойца?",
        "relevant_doc_ids": ["doc_21"],
    },
    {
        "query_id": "q16",
        "query": "Как происходит учет академической успеваемости?",
        "relevant_doc_ids": ["doc_22"],
    },
]

before_update_eval = evaluate_benchmark(extended_benchmark_queries, artifacts=artifacts, top_k=3)
after_update_eval = evaluate_benchmark(extended_benchmark_queries, artifacts=updated_artifacts, top_k=3)

comparison_df = before_update_eval.merge(
    after_update_eval,
    on=["query_id", "query", "relevant_doc_ids"],
    suffixes=("_before", "_after"),
)

display(comparison_df)

summary_comparison_df = pd.DataFrame(
    {
        "metric": ["mean_hit@3", "mean_recall@3", "mean_MRR@3"],
        "before_update": [
            before_update_eval["hit@3"].mean(),
            before_update_eval["recall@3"].mean(),
            before_update_eval["MRR@3"].mean(),
        ],
        "after_update": [
            after_update_eval["hit@3"].mean(),
            after_update_eval["recall@3"].mean(),
            after_update_eval["MRR@3"].mean(),
        ],
    }
)
summary_comparison_df["delta"] = summary_comparison_df["after_update"] - summary_comparison_df["before_update"]
display(summary_comparison_df)

retrieval_before_after_update = comparison_df.copy()
retrieval_before_after_update = retrieval_before_after_update.rename(columns={
    "predicted_doc_ids_before": "before_retrieved_sources",
      "predicted_doc_ids_after": "after_retrieved_sources",
    })
retrieval_before_after_update["changed"] = retrieval_before_after_update["before_retrieved_sources"]!=retrieval_before_after_update["after_retrieved_sources"]
retrieval_before_after_update.to_csv("artifacts/retrieval_before_after_update.csv")

display(Markdown("### Как updated-база отвечает на новые запросы"))
for query in new_queries:
    display(Markdown(f"**Запрос:** {query}"))
    display(search_chunks(query, artifacts=updated_artifacts, top_k=3)[["rank", "score", "doc_id", "title", "chunk_text"]])

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Используем полноценные dense embeddings.
Бэкэнд: SentenceTransformer: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2


,query_id,query,relevant_doc_ids,predicted_doc_ids_before,hit@3_before,recall@3_before,MRR@3_before,first_relevant_rank_before,predicted_doc_ids_after,hit@3_after,recall@3_after,MRR@3_after,first_relevant_rank_after
0,q01,На каком курсе можно поступить в ВУЦ?,doc_01,"doc_18, doc_01, doc_08",1,1.0,0.5,2.0,"doc_18, doc_01, doc_22",1,1.0,0.5,2.0
1,q02,Как получить дополнительные баллы при поступле...,doc_02,"doc_02, doc_12, doc_17",1,1.0,1.0,1.0,"doc_02, doc_12, doc_22",1,1.0,1.0,1.0
2,q03,Есть ли льготы для детей военнослужащих при по...,doc_03,"doc_03, doc_06",1,1.0,1.0,1.0,"doc_03, doc_06",1,1.0,1.0,1.0
3,q04,Сколько лет учатся в ВУЦ?,doc_04,"doc_04, doc_01, doc_18",1,1.0,1.0,1.0,"doc_04, doc_22, doc_01",1,1.0,1.0,1.0
4,q05,"Что будет, если категория годности не подходит...",doc_05,"doc_05, doc_14, doc_15",1,1.0,1.0,1.0,"doc_05, doc_14, doc_15",1,1.0,1.0,1.0
5,q06,Нужно ли потом идти служить после окончания ВУЦ?,doc_06,"doc_06, doc_04, doc_18",1,1.0,1.0,1.0,"doc_06, doc_04, doc_18",1,1.0,1.0,1.0
6,q07,"Придется ли платить, если меня отчислят из ВУЦ?",doc_07,"doc_07, doc_16, doc_06",1,1.0,1.0,1.0,"doc_07, doc_16, doc_06",1,1.0,1.0,1.0
7,q08,Куда подавать заявление на поступление в ВУЦ?,doc_08,"doc_16, doc_08, doc_06",1,1.0,0.5,2.0,"doc_16, doc_08, doc_06",1,1.0,0.5,2.0
8,q09,Что делать если я меняю паспорт во время подач...,doc_09,"doc_09, doc_14",1,1.0,1.0,1.0,"doc_09, doc_22",1,1.0,1.0,1.0
9,q10,Как проходит сдача нормативов по физической по...,doc_10,"doc_20, doc_11, doc_03",0,0.0,0.0,NaN,"doc_20, doc_11, doc_03",0,0.0,0.0,NaN


,metric,before_update,after_update,delta
0,mean_hit@3,0.6875,0.8125,0.125
1,mean_recall@3,0.6875,0.8125,0.125
2,mean_MRR@3,0.5625,0.6875,0.125


### Как updated-база отвечает на новые запросы

**Запрос:** Что такое «Курс молодого бойца» и как он проходит?

,rank,score,doc_id,title,chunk_text
0,1,0.531204,doc_21,Что такое «Курс молодого бойца» и как он прохо...,В соответствии с приказом РТУ МИРЭА прохождени...
1,2,0.419084,doc_17,Принцип формирования списков поступающих в ВУЦ...,Решение о допуске к военной подготовке в Военн...
2,3,0.399403,doc_04,Сколько длится обучение в Военном учебном цент...,семестров);Солдаты запаса — 2 года (4 семестров)


**Запрос:** Как учитывается успеваемость и академическая задолженность?

,rank,score,doc_id,title,chunk_text
0,1,0.719816,doc_22,Как учитывается успеваемость и академическая з...,В целях оценки текущей успеваемости соответств...
1,2,0.582634,doc_22,Как учитывается успеваемость и академическая з...,"курса — за 2-й и 3-й семестры обучения, для ст..."
2,3,0.562067,doc_22,Как учитывается успеваемость и академическая з...,семестры обучения) и занесение результатов в п...


#### Mini-RAG

In [15]:
def build_context_from_retrieval(query: str, artifacts: RetrievalArtifacts, top_k: int = 3) -> Tuple[str, pd.DataFrame]:
    retrieved = search_chunks(query, artifacts=artifacts, top_k=top_k)
    context_blocks = []

    for _, row in retrieved.iterrows():
        block = (
            f"[Источник: {row['doc_id']} | {row['title']} | score={row['score']:.4f}]\n"
            f"{row['chunk_text']}"
        )
        context_blocks.append(block)

    context = "\n\n".join(context_blocks)
    return context, retrieved

query = extended_benchmark_queries[14]["query"]
context, retrieved_df = build_context_from_retrieval(query, artifacts=updated_artifacts, top_k=3)

display(Markdown(f"### Запрос: {query}"))
display(retrieved_df)
print(context)

### Запрос: Что такое Курс молодого бойца?

,rank,score,doc_id,title,chunk_id,chunk_text
0,1,0.723952,doc_21,Что такое «Курс молодого бойца» и как он прохо...,0,В соответствии с приказом РТУ МИРЭА прохождени...
1,2,0.558821,doc_17,Принцип формирования списков поступающих в ВУЦ...,0,Решение о допуске к военной подготовке в Военн...
2,3,0.506173,doc_04,Сколько длится обучение в Военном учебном цент...,1,семестров);Солдаты запаса — 2 года (4 семестров)


[Источник: doc_21 | Что такое «Курс молодого бойца» и как он проходит? | score=0.7240]
В соответствии с приказом РТУ МИРЭА прохождение начальной военной подготовки по программе «Курс молодого бойца» является обязательным. Студенты, не прошедшие курс, к

[Источник: doc_17 | Принцип формирования списков поступающих в ВУЦ (рейтинга)? | score=0.5588]
Решение о допуске к военной подготовке в Военном учебном центре принимается в зависимости от итоговой суммы баллов. Баллы за сдачу физической подготовки

[Источник: doc_04 | Сколько длится обучение в Военном учебном центре РТУ МИРЭА? | score=0.5062]
семестров);Солдаты запаса — 2 года (4 семестров)


In [16]:
def split_into_sentences(text: str) -> List[str]:
    parts = re.split(r"(?<=[.!?])\s+", text.strip())
    return [p.strip() for p in parts if p.strip()]
def generate_answer_from_context(query: str, context: str, max_sentences: int = 2) -> str:
    # Убираем технические строки источников из ранжирования, но не из общего контекста.
    raw_lines = [line.strip() for line in context.splitlines() if line.strip()]
    content_lines = [line for line in raw_lines if not line.startswith("[Источник:")]

    sentence_pool = []
    for line in content_lines:
        sentence_pool.extend(split_into_sentences(line))

    sentence_pool = [s for s in sentence_pool if len(s.split()) >= 4]

    if not sentence_pool:
        return "Недостаточно контекста для построения ответа."

    vectorizer = TfidfVectorizer(ngram_range=(1, 2))
    matrix = vectorizer.fit_transform([query] + sentence_pool).toarray().astype(np.float32)

    query_vec = matrix[0]
    sentence_vecs = matrix[1:]

    query_norm = np.linalg.norm(query_vec) + 1e-12
    sent_norms = np.linalg.norm(sentence_vecs, axis=1) + 1e-12
    scores = (sentence_vecs @ query_vec) / (sent_norms * query_norm)

    ranked_idx = np.argsort(-scores)
    selected_sentences = []
    used_normalized = set()

    for idx in ranked_idx:
        sentence = sentence_pool[idx]
        normalized = sentence.lower().strip()
        if scores[idx] <= 0:
            continue
        if normalized in used_normalized:
            continue
        used_normalized.add(normalized)
        selected_sentences.append(sentence)
        if len(selected_sentences) >= max_sentences:
            break

    if not selected_sentences:
        return "В найденном контексте нет достаточно релевантного фрагмента для уверенного ответа."

    return " ".join(selected_sentences)

In [17]:
answer_example = generate_answer_from_context(query, context)
print(answer_example)

В соответствии с приказом РТУ МИРЭА прохождение начальной военной подготовки по программе «Курс молодого бойца» является обязательным. Студенты, не прошедшие курс, к


In [18]:
def mini_rag_answer(
    query: str,
    artifacts: RetrievalArtifacts,
    top_k: int = 3,
    max_answer_sentences: int = 2,
) -> Dict[str, object]:
    context, retrieved = build_context_from_retrieval(query, artifacts=artifacts, top_k=top_k)
    answer = generate_answer_from_context(query, context=context, max_sentences=max_answer_sentences)

    return {
        "query": query,
        "answer": answer,
        "context": context,
        "sources": retrieved,
    }

In [19]:
rows = []
for query in  extended_benchmark_queries[:5]:
    rag_result = mini_rag_answer(
        query["query"],
        artifacts=updated_artifacts,
        top_k=3,
    )
    display(Markdown(f"### Вопрос: {rag_result['query']}"))
    display(Markdown(f"**Ответ:** {rag_result['answer']}"))
    display(Markdown("**Источники:**"))
    display(rag_result["sources"])
    rows.append({"question":rag_result['query'],
    "answer":rag_result['answer'],
    "retrieved_sources":", ".join(rag_result["sources"]["doc_id"].to_list())
    })
rag_examples = pd.DataFrame(rows)
rag_examples.to_csv("artifacts/rag_examples.csv")


### Вопрос: На каком курсе можно поступить в ВУЦ?

**Ответ:** необходимо поступление в магистратуру РТУ МИРЭА на очную форму обучения. В ВУЦ РТУ МИРЭА поступают студенты 2-го курса очной формы обучения по основной образовательной программе.

**Источники:**

,rank,score,doc_id,title,chunk_id,chunk_text
0,1,0.661932,doc_18,Обязательно ли поступление в магистратуру при ...,1,необходимо поступление в магистратуру РТУ МИРЭ...
1,2,0.652745,doc_01,Когда студент поступает в ВУЦ?,0,В ВУЦ РТУ МИРЭА поступают студенты 2-го курса ...
2,3,0.620367,doc_22,Как учитывается успеваемость и академическая з...,1,"курса — за 2-й и 3-й семестры обучения, для ст..."


### Вопрос: Как получить дополнительные баллы при поступлении?

...

,rank,score,doc_id,title,chunk_id,chunk_text
0,1,0.603813,doc_04,Сколько длится обучение в Военном учебном цент...,1,семестров);Солдаты запаса — 2 года (4 семестров)
1,2,0.580723,doc_22,Как учитывается успеваемость и академическая з...,1,"курса — за 2-й и 3-й семестры обучения, для ст..."
2,3,0.551132,doc_01,Когда студент поступает в ВУЦ?,0,В ВУЦ РТУ МИРЭА поступают студенты 2-го курса ...


### Вопрос: Что будет, если категория годности не подходит (например В)?

**Ответ:** Если по результатам медицинского обследования установлена категория В или Г, а также если у кандидата 4-я категория профессиональной пригодности, то проходить дальнейшие Перезапись возможна, только если кандидат по каким-либо причинам не смог прибыть на место сдачи нормативов и не сдавал их.

**Источники:**

,rank,score,doc_id,title,chunk_id,chunk_text
0,1,0.599935,doc_05,"Что делать, если по результатам медицинского о...",0,Если по результатам медицинского обследования ...
1,2,0.455119,doc_14,Как перезаписаться на сдачу нормативов по физи...,1,"записи. Перезапись возможна, только если канди..."
2,3,0.438865,doc_15,"Категории годности, с которыми граждане допуск...",0,Для допуска к военной подготовке в ВУЦ кандида...
